# Thí nghiệm xoay ảnh — chỉ train trên GPU Colab
Model khởi đầu: det_C_candidate.pt đã train trước đó. Không thay model của app tự động.
Gói input mới: detector_rotation_colab_input.zip (khác ZIP của lượt C trước).

Có 519 ảnh xoay 90/180/270 độ từ nguồn train, chuyển polygon thành box trên bản train,
loại 50 ảnh quarantine và 3 biến thể thuộc nhóm chuối nhãn rỗng đã xác minh.
Val/test gốc giữ nguyên; validation kết hợp val gốc và 301 bản xoay riêng từ nguồn val. Đây là kiểm tra hồi quy, chưa phải test thực tế mới có nhãn sạch.
Ảnh người dùng dùng chẩn đoán KHÔNG đưa vào train.

Ảnh ghép tạo thử chưa đạt chất lượng, đã chặn không đưa vào train.
Notebook chỉ chuẩn bị thí nghiệm: chưa có kết quả độ chính xác GPU cho model mới.


## 1. Chọn GPU và cài thư viện
Trong menu runtime chọn GPU, rồi chạy ô dưới. Máy cá nhân không train.
Không nâng phiên bản Ultralytics giữa một lượt chạy.


In [ ]:
import torch
assert torch.cuda.is_available(), "Chọn GPU runtime trên Colab; không train trên CPU máy cá nhân."
print(torch.cuda.get_device_name(0))
%pip install -q ultralytics==8.4.180


## 2. Mount Drive và giải nén
Upload ZIP vào MyDrive/fruit_rotation/input/detector_rotation_colab_input.zip.
Dữ liệu đọc từ /content; checkpoint ghi trực tiếp Drive. Chạy lại ô này khi runtime bị mất.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, shutil, zipfile, subprocess, sys
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fruit_rotation")
SOURCE = DRIVE / "input/detector_rotation_colab_input.zip"
ROOT = Path("/content/fruit_rotation")
LOCAL_ZIP = Path("/content/detector_rotation_colab_input.zip")
assert SOURCE.is_file(), f"Thiếu ZIP: {SOURCE}"
shutil.copy2(SOURCE, LOCAL_ZIP)
ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

package_hash = sha256(LOCAL_ZIP)
stamp = ROOT / ".package_sha256"
if stamp.exists():
    assert stamp.read_text() == package_hash, "Dataset khác phiên trước; dùng runtime và run mới."
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    for item in archive.infolist():
        assert (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve())
    archive.extractall(ROOT)
stamp.write_text(package_hash)
manifest = json.loads((ROOT / "rotation_manifest.json").read_text())
for relative, expected in manifest["file_hashes"].items():
    assert sha256(ROOT / relative) == expected, f"File sai hash: {relative}"
print("Đã kiểm tra:", len(manifest["file_hashes"]), "file")
print("Lượt ảnh train:", manifest["training_entries"], "| Ảnh xoay:", manifest["rotation_count"])
print("Giới hạn:", manifest["val_test_limitation"])
print("Ảnh ghép không được dùng:", not manifest["synthetic_review_approved"])


## 3. Cấu hình và chạy từng giai đoạn
Lượt đầu ACTION="smoke", chạy ô 3 rồi 4. Hoàn tất 1 epoch thì đổi ACTION="train", chạy lại 3–4.
Sau train: ACTION="val" để so baseline/best/last. Chọn CHECKPOINT theo validation trước.
Sau khi đã chọn checkpoint, ACTION="test" để kiểm tra baseline + checkpoint đã chọn trên test gốc.
Sau đó ACTION="export" để chép best/last thành candidate riêng, không tự thay model app.

Nếu OOM: restart, chạy lại 1–2, giảm BATCH=4.
Bị ngắt: chỉ RESUME=True khi last.pt còn epoch >= 0, optimizer và cùng cấu hình 25 epoch. Giữ RUN_NAME và cùng ZIP.
Nếu lỗi trước checkpoint đầu: đặt RUN_NAME mới và RESUME=False.
Không resume một lượt đã kết thúc.


In [ ]:
if "DRIVE" not in globals():
    raise RuntimeError("Chưa có DRIVE: chạy phần 1–2 trong phiên Colab hiện tại trước.")

ACTION = "smoke"          # smoke → train → val → test → export
RUN_NAME = "rotation_v1"
BATCH = 8
RESUME = False
CHECKPOINT = "best"        # best hoặc last; chốt dựa trên val, trước khi xem test

assert ACTION in {"smoke", "train", "val", "test", "export"}
assert CHECKPOINT in {"best", "last"}
PROJECT = DRIVE / "runs"
print({"action": ACTION, "run": RUN_NAME, "batch": BATCH, "resume": RESUME})


## 4. Thực thi lựa chọn ở phần 3
Cấu hình: YOLO11s, ảnh 640, tối đa 25 epoch, SGD lr0=0.001, xoay nhỏ ±15 độ cộng ảnh xoay 90/180/270,
patience=8, close_mosaic=5, batch8, workers2. Đây là giả thuyết cần đo, không phải model đã được chứng minh tốt hơn.
Giữ checkpoint mỗi epoch trên Drive, thêm snapshot mỗi 5 epoch.
Không dùng test để dò lại tham số; bộ ảnh thực tế mới vẫn cần bổ sung.

Ô này hiện cả stdout/stderr, lưu log tại `MyDrive/fruit_rotation/runs/_logs/` và in lại 80 dòng cuối khi script lỗi. Nếu thiếu biến, chạy lại phần 3; nếu vừa reset runtime, chạy lại 1–2 trước.


In [ ]:
import subprocess, sys
from collections import deque
from datetime import datetime, timezone

required = ("ROOT", "PROJECT", "ACTION", "RUN_NAME", "BATCH", "RESUME", "CHECKPOINT")
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Thiếu biến " + ", ".join(missing) + ". Chạy phần 1–2 nếu vừa đổi runtime, rồi phần 3 trước phần 4.")
assert ACTION in {"smoke", "train", "val", "test", "export"}, "ACTION không hợp lệ; sửa ở phần 3."
assert CHECKPOINT in {"best", "last"}, "CHECKPOINT phải là best hoặc last."

script_mode = "evaluate" if ACTION in {"val", "test"} else ACTION
command = [sys.executable, "-u", str(ROOT / "train_rotation.py"), script_mode,
           "--root", str(ROOT), "--project", str(PROJECT), "--name", RUN_NAME,
           "--batch", str(BATCH), "--checkpoint", CHECKPOINT]
if ACTION in {"val", "test"}:
    command += ["--split", ACTION]
if RESUME:
    assert ACTION in {"smoke", "train"}, "RESUME chỉ dùng cho lượt train bị ngắt."
    import torch
    resume_dir = PROJECT / (RUN_NAME + ("_smoke" if ACTION == "smoke" else ""))
    resume_checkpoint = torch.load(resume_dir / "weights/last.pt", map_location="cpu", weights_only=False)
    saved_epoch = resume_checkpoint.get("epoch", -1)
    saved_epochs = resume_checkpoint.get("train_args", {}).get("epochs")
    if saved_epoch is None or saved_epoch < 0 or resume_checkpoint.get("optimizer") is None:
        raise RuntimeError("Checkpoint đã kết thúc hoặc thiếu optimizer: không resume. Nếu lượt đã train xong, đặt ACTION=val và RESUME=False ở phần 3.")
    expected_epochs = 1 if ACTION == "smoke" else 25
    if saved_epochs != expected_epochs or saved_epoch + 1 >= saved_epochs:
        raise RuntimeError(f"Checkpoint: {saved_epoch + 1} epoch đã xong, mục tiêu {saved_epochs}; cấu hình cần {expected_epochs}. Kiểm tra lượt cũ trước khi tiếp tục.")
    print(f"Resume đã kiểm tra: {saved_epoch + 1}/{saved_epochs} epoch hoàn tất", flush=True)
    del resume_checkpoint
    command += ["--resume"]
# Hiện cả stdout/stderr và in lại lỗi gốc ở cuối.
log_dir = PROJECT / "_logs"
log_dir.mkdir(parents=True, exist_ok=True)
log_path = log_dir / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + ACTION + ".log")
tail = deque(maxlen=80)
print("Log đầy đủ trên Drive:", log_path, flush=True)
with log_path.open("w", encoding="utf-8") as log:
    log.write("Command: " + repr(command) + "\n")
    log.flush()
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
                tail.append(line)
            return_code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
if return_code:
    print("\n===== LỖI GỐC / 80 DÒNG CUỐI =====", flush=True)
    print("".join(tail), flush=True)
    raise RuntimeError(f"Script dừng với mã {return_code}. Gửi phần LỖI GỐC bên trên; log: {log_path}")

if ACTION == "smoke":
    print("Smoke xong: đổi ACTION=train, RESUME=False rồi chạy lại phần 3–4.")
elif ACTION == "train":
    print("Train xong: đổi ACTION=val, RESUME=False rồi chạy lại phần 3–4.")
elif ACTION == "val":
    print("Xem comparison_val.json trên Drive, chốt CHECKPOINT rồi mới ACTION=test.")
elif ACTION == "test":
    print("Xem comparison_test.json. Chỉ nhận model nếu đạt các tiêu chí đã chốt; cần thử ảnh thực tế.")
else:
    print("Tải candidate trực tiếp từ Drive:", PROJECT / RUN_NAME / "candidate")


## Kết quả trên Drive
MyDrive/fruit_rotation/runs/rotation_v1/weights/{best,last}.pt
MyDrive/fruit_rotation/runs/rotation_v1/comparison_val.json
MyDrive/fruit_rotation/runs/rotation_v1/comparison_test.json
MyDrive/fruit_rotation/runs/rotation_v1/candidate/det_rotation_{best,last}.pt

Cần xem mAP, AP chuối/táo/cam, số ảnh nền báo sai và ảnh nhiều quả mới.
Mức confidence 0.40 của kiểm tra nền ở notebook này khác phép đo conf0.25 của bàn giao C cũ.
So với baseline được đo lại trong cùng báo cáo. Không mặc định nhận model chỉ vì loss giảm.
Classifier giữ nguyên, model C và candidate cũ vẫn được lưu.
